# Day 10 — Renk Uzayları ve Renk Farkı
## BGR, HSV, CIE L*a*b* Uzayları ve Delta E (CIE76) Boya Partisi Kalite Analizi

> **Aşama:** Faz 2 — Bilgisayarlı Görü (Day 09–15)
> **Resmi Staj Defteri Konusu:** Renk Uzayları ve Renk Farkı (Yaprak 19 & 20)

### 1. Problem
Tekstil üretiminde boya partileri (dye-lot) arasında gözle fark edilemeyen küçük renk sapmaları, dokuma halı üzerinde bir araya geldiğinde belirgin çizgilere (barre hatası) neden olur. Standart RGB uzayındaki mesafeler insan gözünün algısal duyarlılığıyla doğrusal orantılı değildir.

### 2. Why the Problem Matters
CIE L*a*b* uzayı insan görüş sistemine göre algısal olarak düzgün (perceptually uniform) tasarlanmıştır. $\Delta E$ renk farkı metriği, tolerans eşiğini ($\Delta E \le 2.0$) aşan iplik bobinlerinin üretime girmesini engeller.

### 3. Engineering Concepts
- **HSV Uzayı**: Renk Özü (Hue), Doygunluk (Saturation) ve Parlaklık (Value) - aydınlatma değişimlerine karşı dayanıklı ayrıştırma.
- **CIE L*a*b* Uzayı**: $L^*$ açıklık, $a^*$ yeşil-kırmızı ekseni, $b^*$ mavi-sarı ekseni.
- **CIE $\Delta E_{76}$**: Algısal renk farkı: $\Delta E = \sqrt{(\Delta L^*)^2 + (\Delta a^*)^2 + (\Delta b^*)^2}$.

In [ ]:
# 4. Library / API Investigation
import cv2
import numpy as np
from day10.mini_project.src.color_difference import ColorDifferenceAnalyzer, bgr_to_cielab

ref_bgr = np.array([30, 80, 180], dtype=np.uint8)  # Sıcak kiremit tonu
sample_bgr = np.array([32, 82, 177], dtype=np.uint8)
analyzer = ColorDifferenceAnalyzer(tolerance_threshold=2.0)
de, grade, ok = analyzer.grade_color_match(ref_bgr, sample_bgr)
print(f"Delta E: {de} | Değerlendirme: {grade} | Kabul: {ok}")

In [ ]:
# 5. Minimal Implementation
lab_ref = bgr_to_cielab(ref_bgr)
lab_sample = bgr_to_cielab(sample_bgr)
print("Referans L*a*b*:", np.round(lab_ref, 2))
print("Numune L*a*b*:", np.round(lab_sample, 2))

In [ ]:
# 6. Experiment: Farklı Renk Tonları Delta E Testi
test_samples = [
    ("Uyumlu İplik A", np.array([31, 81, 179], dtype=np.uint8)),
    ("Sınırda İplik B", np.array([35, 87, 170], dtype=np.uint8)),
    ("Hatalı İplik C", np.array([60, 120, 140], dtype=np.uint8)),
]
for name, s_bgr in test_samples:
    d_val, g_text, is_acc = analyzer.grade_color_match(ref_bgr, s_bgr)
    print(f"{name}: Delta E={d_val:.2f} -> {g_text}")

In [ ]:
# 7. Visualization: Delta E Tolerans Grafiği
import matplotlib.pyplot as plt

names = [t[0] for t in test_samples]
de_vals = [analyzer.grade_color_match(ref_bgr, t[1])[0] for t in test_samples]

plt.figure(figsize=(6, 3.5))
bars = plt.bar(names, de_vals, color=["#2ca02c", "#ff7f0e", "#d62728"], width=0.4)
plt.axhline(2.0, color="black", linestyle="--", label="Kabul Edilebilir Eşik (ΔE = 2.0)")
plt.ylabel("CIE Delta E")
plt.title("İplik Boya Partisi Renk Sapma Analizi")
plt.legend()
plt.grid(axis="y", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# 8. Validation
assert de_vals[0] <= 2.0
assert de_vals[2] > 5.0
print("Tüm renk tolerans sınırları başarıyla doğrulandı.")

In [ ]:
# 9. Failure Cases: Tamamen aynı renkte Delta E = 0 kontrolü
same_de, _, _ = analyzer.grade_color_match(ref_bgr, ref_bgr)
assert same_de == 0.0
print("Özdeş renk girdisinde Delta E sıfır olarak doğrulandı.")

### 10. Conclusions
CIE L*a*b* renk uzayında Delta E metriği kullanılarak dokuma ipliklerindeki boya partisi farklılıkları algısal hassasiyetle derecelendirilmiştir.